# 04 — Generalization, Robustness, Uncertainty and Progressive Temporal Proof of Principle

**Questions:** How sensitive are held-out estimates to correlated frames and source identity? Which mistakes matter operationally? Does adding labelled historical video data improve classification on later frames?

This notebook retains source-video bootstrap uncertainty, error analysis and grayscale pairing, then performs the newly agreed **two-sequence progressive ResNet18 study**. All tables and plots are produced by execution, not inserted as measured values in advance.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage
import src as mr
from src.temporal import run_progressive_temporal
ROOT=Path.cwd().resolve()
settings=mr.load_settings(ROOT)
device=mr.setup(settings['seed'])
print('Device:',device,'| Source bootstrap resamples:',settings['bootstrap_resamples'])


## ResNet18 uncertainty across all four regimes

The difference between a score and its uncertainty matters. This cell loads frozen test predictions from Notebook 03, resamples **source groups with replacement**, recalculates Macro F1 and saves numerical intervals and a reproducible figure.

In [ ]:
bootstrap_rows=[];all_draws={};error_rows=[]
for regime in mr.REGIMES:
    p=pd.read_csv(Path(settings['out'])/'resnet18'/regime/'heldout_test_predictions.csv')
    r,draws=mr.source_bootstrap(p,seed=settings['seed'],reps=settings['bootstrap_resamples'])
    bootstrap_rows.append(dict(regime=regime,**r));all_draws[regime]=draws
    error_rows.append(dict(regime=regime,**mr.classify_errors(p)))
bootstrap=pd.DataFrame(bootstrap_rows);errors=pd.DataFrame(error_rows)
bootstrap.to_csv(Path(settings['out'])/'bootstrap_resnet18.csv',index=False)
errors.to_csv(Path(settings['out'])/'resnet18_error_summary.csv',index=False)
display(bootstrap);display(errors)

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(4)
lower=bootstrap.point_macro_f1-bootstrap.ci_lower
upper=bootstrap.ci_upper-bootstrap.point_macro_f1
ax.errorbar(x,bootstrap.point_macro_f1,yerr=np.vstack([lower,upper]),fmt='o',capsize=5)
ax.set_xticks(x,bootstrap.regime);ax.set_ylim(0,1.05)
ax.set(title='ResNet18 Held-out Macro F1 with Source-video Bootstrap 95% CI',ylabel='Macro F1')
fig.tight_layout();path=Path(settings['out'])/'figures'/'resnet18_source_video_bootstrap.png';fig.savefig(path,dpi=150);plt.show()

## Paired RGB-versus-grayscale source bootstrap

For each selected regime, join RGB and grayscale predictions on **exact sample IDs**. Within each bootstrap draw, sample the same source groups for both models. The output estimates the grayscale-minus-RGB **held-out Macro F1 difference**, not tuning F1. A confidence interval containing zero does not establish equivalence; it means the observed difference is not resolved by this interval.

In [ ]:
color_rows=[]
for regime in ('O','ST'):
    rgb=pd.read_csv(Path(settings['out'])/'scratch'/regime/'heldout_test_predictions.csv')
    gray=pd.read_csv(Path(settings['out'])/'grayscale'/regime/'heldout_test_predictions.csv')
    r=mr.paired_source_bootstrap(rgb,gray,seed=settings['seed'],reps=settings['bootstrap_resamples'])
    color_rows.append(dict(regime=regime,**r))
color=pd.DataFrame(color_rows);color.to_csv(Path(settings['out'])/'paired_color_bootstrap.csv',index=False)
display(color)

## Failure modes and confidence

False negatives are real `flip` frames predicted `notflip`; false positives are `notflip` frames predicted `flip`. A high-confidence error means a wrong prediction with confidence above 0.90. These quantities are **counted from individual predictions**. The analysis does not assume that T or ST must contain only false negatives.

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(4);a=ax.bar(x-.20,errors.fp,width=.4,label='False positive');b=ax.bar(x+.20,errors.fn,width=.4,label='False negative')
ax.set_xticks(x,errors.regime);ax.set(ylabel='Errors',title='ResNet18 Held-out False Positives and False Negatives');ax.legend()
fig.tight_layout();path=Path(settings['out'])/'figures'/'resnet18_failure_analysis.png';fig.savefig(path,dpi=150);plt.show()
print('Notebook 04 completed without manually supplied performance numbers.')

## Original generalization and uncertainty evidence — reference only

The archived bootstrap and failure tables preserve the earlier study. The new source-level bootstrap and progressive temporal evaluation remain separate experiments.

In [ ]:
historical_bootstrap=mr.read_archived_result_table(ROOT/'submission'/'results'/'bootstrap_summary.csv')
historical_errors=mr.read_archived_result_table(ROOT/'submission'/'results'/'final_error_summary.csv')
display(historical_bootstrap[['regime','bootstrap_level','point_estimate','ci_lower','ci_upper','n_resamples']])
display(historical_errors)

## Progressive T/ST experiment — methods (draft prepared before execution)

**Practical objective.** Estimate whether more chronologically available, **labelled** historical frames improve frame-wise page-flip classification, and whether performance differs for increasingly distant future image positions.

**Model and procedure.** Start two ImageNet-pretrained ResNet18 models: **T**, training on early frames of each familiar source; and **ST**, training on early frames from known sources only, with the predeclared ENV-03 sources entirely withheld. Train at the 25% checkpoint with the established two-stage configuration, then **continue fine-tuning the same model** as prefixes grow to 50%, 75% and 90%. No new Bayesian hyperparameter sweep is performed. A chronological validation tail within each available prefix supports checkpoint selection. The evaluation suffix is never used for that checkpoint's optimization.

**Complementary analyses.** At each checkpoint, compute (1) one **overall future Macro F1** over all remaining frames, and (2) **horizon-window Macro F1 and flip recall**, pooling equivalent future positions across sources in three-frame windows. A window is reportable only with at least 20 frames per class and at least eight sources per class. Report class support and source counts, with no score for unsupported windows. To compare training coverage on the same examples, also evaluate the common final 10% of every eligible recording.

**Interpretation restrictions.** T tests later frames of familiar sources. ST tests later positions in **unseen recordings**: this is not decay since the model last observed the tested source. Frame numbers do not provide known time intervals or dates. These are retrospective supervised model updates that require labels; they do **not** establish online learning, real-time streaming efficiency, or forecasting of a future event before its frame is observed.

The cell below produces the measured tables, predictions, checkpoints, histories and plots, stored independently of the original T/ST model comparison. Figures are not available until it has run.


In [ ]:
manifest=pd.read_csv(Path(settings['out'])/'canonical_manifest.csv')
environment_map=ROOT/'manifests'/'video_environment_map.csv'
overall_future,horizon_windows=run_progressive_temporal(
    manifest,settings,device,environment_map
)
display(overall_future)
display(horizon_windows[['regime','coverage_percent','start_horizon','end_horizon',
                         'n_frames','n_flip','n_notflip','n_sources','reportable',
                         'macro_f1','flip_recall']])
for regime in ('T','ST'):
    p=Path(settings['out'])/'progressive_temporal'/'figures'
    display(IPyImage(filename=str(p/f'{regime}_overall_future.png')))
    display(IPyImage(filename=str(p/f'{regime}_horizon_macro_f1.png')))


### Draft results interpretation — complete after execution

Use the overall-future table to assess whether progressive data availability improves classification across the *entire* future suffix. Then inspect the horizon curves for changes as future-frame position increases. Finally compare the common-final-10% scores to separate the data-quantity comparison from the changing evaluation suffix. Treat small differences cautiously, state the number of contributing sources and class observations, and avoid attributing changes to elapsed-time drift.

**Operational interpretation to complete using results:** Does adding more labelled frames plausibly justify scheduled model updates? Are missed flips (false negatives) reduced? Does a shorter update interval appear useful? These questions can support a future deployment pilot, but no update schedule is established until the actual measurements are reviewed.
